# Plotting with Plotly

In [ ]:
# Cell hidden on the website (remove-cell in tags)
# https://github.com/holoviz/holoviews/pull/6391
import warnings

warnings.filterwarnings("ignore", message="When grouping with a length-1 list-like", category=FutureWarning)

This page demonstrates the use of the **Plotly** plotting backend, the equivalent page demonstrating the Matplotlib backend may be found [here](Plotting_with_Matplotlib.ipynb)

As we discovered in the [Introduction](Introduction.ipynb), HoloViews allows plotting a variety of data types. Here we will use the sample data module and load the pandas and dask hvPlot API:

In [ ]:
import hvplot.pandas  # noqa
import hvplot.dask  # noqa

hvplot.extension('plotly')

As we learned the hvPlot API closely mirrors the [Pandas plotting API](https://pandas.pydata.org/pandas-docs/stable/visualization.html), but instead of generating static images when used in a notebook, it uses HoloViews to generate either static or dynamically streaming Bokeh plots. Static plots can be used in any context, while streaming plots require a live [Jupyter notebook](https://jupyter.org), a deployed [Bokeh Server app](https://bokeh.pydata.org/en/latest/docs/user_guide/server.html), or a deployed [Panel](https://panel.holoviz.org) app.

HoloViews provides an extensive, very rich set of objects along with a powerful set of operations to apply, as you can find out in the [HoloViews User Guide](https://holoviews.org/user_guide/index.html). But here we will focus on the most essential mechanisms needed to make your data visualizable, without having to worry about the mechanics going on behind the scenes.

We will be focusing on three different datasets:

- A small CSV file of Apple Inc. (AAPL) daily stock prices
- A dataset of Palmer Penguins measurements used for statistical plots
- A larger dataset of NYC taxi trips with pickup and dropoff locations in Web Mercator coordinates

The ``hvplot.sampledata`` module provides access to these datasets via the ``hvsampledata`` package, which we can load either using pandas:

In [ ]:
from hvplot.sampledata import apple_stocks, nyc_taxi_remote, penguins

apple = apple_stocks('pandas')
print(type(apple))
apple.head()

Or using dask as a ``dask.DataFrame``:

In [ ]:
taxi = nyc_taxi_remote('dask', lazy=True)
print(type(taxi))
taxi.head()

## The plot interface

The ``dask.dataframe.DataFrame.hvplot`` and ``pandas.DataFrame.hvplot`` interfaces (and Series equivalents) from HvPlot provide a powerful high-level API to generate complex plots. The ``.hvplot`` API can be called directly or used as a namespace to generate specific plot types.

### The plot method

The most explicit way to use the plotting API is to specify the names of columns to plot on the ``x``- and ``y``-axis respectively:

In [ ]:
apple.hvplot.line(x='date', y='close')

As you'll see in more detail below, you can choose which kind of plot you want to use for the data:

In [ ]:
apple.hvplot(x='date', y='close', kind='scatter')

To group the data by one or more additional columns, specify an additional ``by`` variable. As an example here we will plot pickup locations, grouping by the number of passengers (``passenger_count``). We will sample 1% of the dataset and filter to trips with 1 or 2 passengers to keep two groups for a readable plot:

In [ ]:
taxi_subset = taxi[taxi.passenger_count.isin([1, 2])].sample(frac=0.01).compute()
taxi_subset.hvplot(x='pickup_x', y='pickup_y', by='passenger_count', kind='scatter')

Here we have specified the `x` axis explicitly, which can be omitted if the Pandas index column is already the desired x axis. Similarly, here we specified the `y` axis; by default all of the non-index columns would be plotted (which would be a lot of data in this case). If you don't specify the 'y' axis, it will have a default label named 'value', but you can then provide a y axis label explicitly using the ``value_label`` option.

Putting all of this together we will plot the closing price, daily high and daily low on the y-axis, specifying 'date' as the x, and relabel the y-axis to display the 'Price (USD)'.

In [ ]:
apple.hvplot(x='date', y=['open', 'close'], value_label='Price (USD)')

### The hvplot namespace

Instead of using the ``kind`` argument to the plot call, we can use the ``hvplot`` namespace, which lets us easily discover the range of plot types that are supported. Use tab completion to explore the available plot types:

```python
apple.hvplot.<TAB>
```

Plot types available include:

* <a href="#area">``.area()``</a>: Plots a  area chart similar to a line chart except for filling the area under the curve and optionally stacking 
* <a href="#bars">``.bar()``</a>: Plots a bar chart that can be stacked or grouped
* <a href="#bivariate">``.bivariate()``</a>: Plots 2D density of a set of points 
* <a href="#box-whisker-plots">``.box()``</a>: Plots a box-whisker chart comparing the distribution of one or more variables
* <a href="#heatmap">``.heatmap()``</a>: Plots a heatmap to visualizing a variable across two independent dimensions
* <a href="#hexbins">``.hexbins()``</a>: Plots hex bins
* <a href="#histogram">``.hist()``</a>: Plots the distribution of one or histograms as a set of bins
* <a href="#kde-density">``.kde()``</a>: Plots the kernel density estimate of one or more variables.
* <a href="#the-plot-method">``.line()``</a>: Plots a line chart (such as for a time series)
* <a href="#scatter">``.scatter()``</a>: Plots a scatter chart comparing two variables
* <a href="#step">``.step()``</a>: Plots a step chart akin to a line plot
* <a href="#tables">``.table()``</a>: Generates a SlickGrid DataTable
* <a href="#groupby">``.violin()``</a>: Plots a violin plot comparing the distribution of one or more variables using the kernel density estimate

#### Area

Like most other plot types the ``area`` chart supports the three ways of defining a plot outlined above. An area chart is most useful when plotting multiple variables in a stacked chart. This can be achieve by specifying ``x``, ``y``, and ``by`` columns or using the ``columns`` and ``index``/``use_index`` (equivalent to ``x``) options:

In [ ]:
apple.hvplot.area(x='date', y=['high', 'low'])

We can also explicitly set ``stacked`` to False to compare the values directly:

In [ ]:
apple.hvplot.area(x='date', y=['high', 'low'], stacked=False)

Another use for an area plot is to visualize the spread of a value over time. For instance using the apple dataset we may want to see the range of monthly mean closing prices within each year. For that purpose we group by year and month to get the monthly mean close, and then compute the min/max of those monthly means per year. Since the output of ``hvplot`` is just a regular holoviews object, we can use the overlay operator

In [ ]:
apple['year'] = apple['date'].dt.year
apple['month'] = apple['date'].dt.month
monthly = apple.groupby(['year', 'month'])['close'].mean().reset_index()
y_min_max = monthly.groupby('year')['close'].agg(['min', 'max'])
y_mean = monthly.groupby('year')['close'].mean()

y_min_max.hvplot.area(x='year', y='min', y2='max') * y_mean.hvplot()

#### Bars

In the simplest case we can use ``.hvplot.bar`` to plot ``x`` against ``y``. We'll use ``rot=90`` to rotate the tick labels on the x-axis making the dates easier to read:

In [ ]:
apple_subset = apple[:50]
apple_subset.hvplot.bar(x='date', y='close', rot=90)

If we want to compare multiple columns instead we can set ``y`` to a list of columns. Using the ``stacked`` option we can then compare the column values more easily:

In [ ]:
apple_subset.hvplot.bar(x='date', y=['open', 'close'],
                 stacked=True, rot=90, width=800, legend='top_left')

#### Scatter

The scatter plot supports many of the same features as the other chart types we have seen so far but can also be colored by another variable using the ``c`` option. 

In [ ]:
penguins = penguins('pandas')
penguins.hvplot.scatter(x='bill_length_mm', y='bill_depth_mm', c='body_mass_g')

Anytime that color is being used to represent a dimension, the ``cmap`` option can be used to control the colormap that is used to represent that dimension. Additionally, the colorbar can be disabled using ``colorbar=False``.

#### Step

A step chart is very similar to a line chart but instead of linearly interpolating between samples the step chart visualizes discrete steps. The point at which to step can be controlled via the ``where`` keyword allowing `'pre'`, `'mid'` (default) and `'post'` values:

In [ ]:
apple_subset.hvplot.step(x='date', y=['high', 'low'])

#### HexBins

You can create hexagonal bin plots with the ``hexbin`` method. Hexbin plots can be a useful alternative to scatter plots if your data are too dense to plot each point individually.

In [ ]:
# penguins.hvplot.hexbin(x='bill_length_mm', y='bill_depth_mm')

<div class="alert alert-warning" role="alert">
    HexBins plots <a href='https://github.com/holoviz/holoviews/issues/5219'>not yet supported</a> with the Plotly backend.
</div>

#### Bivariate

You can create a 2D density plot with the ``bivariate`` method. Bivariate plots can be a useful alternative to scatter plots if your data are too dense to plot each point individually.

In [ ]:
# penguins.hvplot.bivariate(x='bill_length_mm', y='bill_depth_mm')

<div class="alert alert-warning" role="alert">
Commented as this is <a href='https://github.com/holoviz/holoviews/issues/5220'>not currently supported</a> with the Plotly backend and raises an error.
</div>

#### HeatMap

A ``HeatMap`` lets us view the relationship between three variables, so we specify the 'x' and 'y' variables and an additional 'C' variable. Additionally we can define a ``reduce_function`` that computes the values for each bin from the samples that fall into it. Here we compute pairwise correlations between the penguins measurement columns, reshape the correlation matrix to long form, and plot the correlation coefficient for each variable pair:

In [ ]:
corr = penguins[[c for c in penguins.columns if c.split("_")[-1] in ("mm", "g")]].corr()
# Convert to long-form for heatmap
corr_df = corr.stack().reset_index()
corr_df.columns = ['variable_1', 'variable_2', 'correlation']

corr_df.hvplot.heatmap(
    x='variable_1',
    y='variable_2',
    C='correlation',
    cmap='coolwarm',
    clim=(-1, 1),
    title='Correlation Heatmap',
)

#### Tables

Unlike all other plot types, a table only supports one signature: either all columns are plotted, or a subset of columns can be selected by defining the ``columns`` explicitly:

In [ ]:
apple.hvplot.table(columns=['date', 'close', 'volume'], width=400)

### Distributions

Plotting distributions differs slightly from other plots since they plot only one variable in the simple case rather than plotting two or more variables against each other. Therefore when plotting these plot types no ``index`` or ``x`` value needs to be supplied. Instead:

1. Declare a single ``y`` variable, e.g. ``source.plot.hist(variable)``, or
2. Declare a ``y`` variable and ``by`` variable, e.g. ``source.plot.hist(variable, by='Group')``, or
3. Declare columns or plot all columns, e.g. ``source.plot.hist()`` or ``source.plot.hist(columns=['A', 'B', 'C'])``

#### Histogram

The Histogram is the simplest example of a distribution; often we simply plot the distribution of a single variable, in this case the closing price. Additionally we can define a range over which to compute the histogram and the number of bins using the ``bin_range`` and ``bins`` arguments respectively:

In [ ]:
apple.hvplot.hist(y='close')

Or we can plot the distribution of multiple columns:

In [ ]:
columns = ['close', 'high', 'low']
apple.hvplot.hist(y=columns, bins=50, legend='top', height=400)

We can also group the data by another variable. Here we'll use ``subplots`` to split each cluster category out into its own plot:

In [ ]:
apple.hvplot.hist(y=columns, legend='top', height=400, subplots=True)

#### KDE (density)

You can also create density plots using ``hvplot.kde()`` or ``hvplot.density()``:

In [ ]:
apple.hvplot.kde(y='close')

Comparing the distribution of multiple columns is also possible:

In [ ]:
columns=['close', 'high', 'low']
apple.hvplot.kde(y=columns, value_label='Price (USD)', legend='top_right')

The ``hvplot.kde`` also supports the ``by`` keyword:

In [ ]:
penguins.hvplot.kde('body_mass_g', by='species', width=300, subplots=True)

#### Box-Whisker Plots

Just like the other distribution-based plot types, the box-whisker plot supports plotting a single column:

In [ ]:
penguins.hvplot.box(y='body_mass_g')

It also supports multiple columns and the same options as seen previously (``legend``, ``invert``, ``value_label``):

In [ ]:
columns=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
penguins.hvplot.box(y=columns, legend=False, group_label='Body Part', value_label='Size (mm)', invert=True)

Lastly, it also supports using the ``by`` keyword to split the data into multiple subsets:

In [ ]:
penguins.hvplot.box(y='body_mass_g', by='species')

## Composing Plots

One of the core strengths of HoloViews is the ease of composing
different plots. Individual plots can be composed using the ``*`` and
``+`` operators, which overlay and compose plots into layouts
respectively. For more information on composing objects, see the
HoloViews [User Guide](https://holoviews.org/user_guide/Composing_Elements.html).

By using these operators we can combine multiple plots into composite plots. A simple example is overlaying two plot types:

In [ ]:
apple_subset.hvplot(x='date', y='close') * apple_subset.hvplot.scatter(x='date', y='close', c='k')

We can also lay out different plots and tables together:

In [ ]:
(apple_subset.hvplot.bar(x='date', y='close', rot=90, width=550) +
 apple_subset.hvplot.table(['date', 'close', 'volume'], width=420))

## Large data

The previous examples summarized the fairly large NYC taxi dataset using statistical plot types that aggregate the data into a feasible subset for plotting.  We can instead aggregate the data directly into the viewable image using [datashader](https://datashader.org), which provides a rendering of the entire set of raw data available (as far as the resolution of the screen allows). Here we plot the pickup locations, aggregating by the sum of passenger counts:

In [ ]:
import datashader as ds

taxi.hvplot.points('pickup_x', 'pickup_y', rasterize=True, dynspread=True,
                   aggregator=ds.sum('passenger_count'),
                   cnorm='eq_hist', cmap='fire')

## Groupby

Thanks to the ability of HoloViews to explore a parameter space with a set of widgets we can apply a groupby along a particular column or dimension. For example we can view the distribution of the penguins body mass by species grouped by year, allowing the user to choose which value group to display:

In [ ]:
penguins.hvplot.violin(y='body_mass_g', by='species', groupby='year', height=300)

This user guide merely provided an overview over the available plot types; to see a detailed description on how to customize plots see the [Plotting Options](../ref/plotting_options/index.md) reference.